# 第2章：栈的表达式求值实验 — 动手实验

## 小节概述

本小节是第2章的核心实践环节。你将在昇腾 NPU 上完成3个基于栈结构的 Ascend C 自定义算子的编译、部署和精度验证：

- **BracketMatchLite**：括号匹配检验
- **SuffixEvalLite**：后缀表达式求值
- **InfixToPostfixLite**：中缀转后缀

**前置要求**：已完成 02.01 章节介绍，了解栈的基本概念和 Ascend C Kernel 的基本结构。

**学习目标**：
1. 掌握 Ascend C 自定义算子的完整编译流程（msopgen 生成 → Kernel 编写 → 编译打包 → 安装部署）。
2. 理解 Benchmark Runner 如何通过 ACLNN API 调用自定义算子。
3. 验证3个算子在 910B3 NPU 上的精度正确性。

**实验目录**：`src/stack_expr_lab/`

## 步骤1：确认环境和目标平台

In [ ]:
import os

ASCEND_HOME = os.environ.get('ASCEND_HOME_PATH', '/usr/local/Ascend/ascend-toolkit/latest')
assert os.path.exists(ASCEND_HOME), f'CANN SDK 未安装: {ASCEND_HOME}'
print(f'CANN SDK 路径: {ASCEND_HOME}')

# 根据实际 NPU 型号设置 TARGET
# Ascend910 (A3): ascend910_93
# Ascend910B:     ascend910b
# Ascend310B1:    ascend310b
TARGET = os.environ.get('TARGET', 'ascend910b')
print(f'目标平台: {TARGET}')

NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
if not os.path.exists(os.path.join(NOTEBOOK_DIR, 'src', 'stack_expr_lab')):
    NOTEBOOK_DIR = os.getcwd()
LAB_DIR = os.path.join(NOTEBOOK_DIR, 'src', 'stack_expr_lab')
assert os.path.exists(LAB_DIR), f'实验目录不存在: {LAB_DIR}'
print(f'实验目录: {LAB_DIR}')
print('环境检查通过')

## 步骤2：编译3个自定义算子（约2-3分钟）

编译过程包括：
1. 使用 `msopgen` 生成算子工程模板（3个算子逐个生成）
2. 替换 Kernel 实现代码
3. 编译生成算子包（.run 安装包）

In [ ]:
import subprocess

source_env = f'source {ASCEND_HOME}/set_env.sh'
result = subprocess.run(
    f'{source_env} && TARGET={TARGET} bash scripts/build_ops.sh',
    shell=True, cwd=LAB_DIR, capture_output=True, text=True, executable='/bin/bash'
)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
else:
    print('算子编译成功')

## 步骤3：编译 Benchmark Runner

In [ ]:
source_env = f'source {ASCEND_HOME}/set_env.sh'
result = subprocess.run(
    f'{source_env} && source scripts/env_custom_opp.sh && bash scripts/build_runner.sh',
    shell=True, cwd=LAB_DIR, capture_output=True, text=True, executable='/bin/bash'
)
print(result.stdout[-300:] if len(result.stdout) > 300 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-300:])
else:
    print('Runner 编译成功')

## 步骤4：生成测试数据

In [ ]:
result = subprocess.run(
    'python3 scripts/gen_data.py',
    shell=True, cwd=LAB_DIR, capture_output=True, text=True
)
print(result.stdout)
assert os.path.exists(os.path.join(LAB_DIR, 'data/input/bracket_input.bin')), '测试数据生成失败'
print('测试数据生成成功')

## 步骤5：运行 Benchmark 验证3个算子

In [ ]:
source_env = f'source {ASCEND_HOME}/set_env.sh'
result = subprocess.run(
    f'{source_env} && source scripts/env_custom_opp.sh && aclnn_runner/build/main_stack_benchmark data',
    shell=True, cwd=LAB_DIR, capture_output=True, text=True, executable='/bin/bash'
)
print(result.stdout)

pass_count = sum(1 for line in result.stdout.splitlines() if line.strip() == 'PASS')
print(f'\n=== 验证结果: {pass_count}/3 PASS ===')
assert pass_count == 3, f'期望3个PASS，实际{pass_count}个'
print('所有算子验证通过')

## 算子源码解读

### BracketMatchLite Kernel 核心逻辑

```cpp
class KernelBracketMatchLite {
    // ...
    __aicore__ inline void Process() {
        uint32_t top = 0;       // 栈顶指针
        for (uint32_t i = 0; i < exprLength; i++) {
            char ch = (char)xGm.GetValue(start + i);
            if (ch == '(' || ch == '[' || ch == '{') {
                ubStack[top++] = ch;          // Push
            } else if (ch == ')' || ch == ']' || ch == '}') {
                if (top == 0) { status = 1; break; }  // 失配①
                char expected = ubStack[--top];         // Pop
                if (!match(expected, ch)) { status = 2; break; }  // 失配②
            }
        }
        if (status == 0 && top != 0) status = 3;  // 失配③
    }
private:
    char ubStack[MAX_STACK_SIZE];  // Local Memory 上的栈空间
};
```

### SuffixEvalLite Kernel 核心逻辑

```cpp
// 后缀表达式求值：操作数栈
float ubOpndStack[MAX_OPND_STACK];  // Local Memory
uint32_t top = 0;

for (uint32_t i = 0; i < tokenCount; i++) {
    int32_t token = tokens[start + i];
    if (token >= 0) {  // 操作数（非负整数编码）
        ubOpndStack[top++] = (float)token;  // Push
    } else {  // 运算符（负数编码：-1=+, -2=-, -3=*, -4=/）
        float b = ubOpndStack[--top];  // Pop
        float a = ubOpndStack[--top];  // Pop
        ubOpndStack[top++] = operate(a, token, b);  // Push result
    }
}
result = ubOpndStack[0];  // 最终结果
```

## 实验总结

你已经完成了：

1. 编译3个 Ascend C 自定义算子
2. 编译 Benchmark Runner
3. 生成测试数据
4. 运行 benchmark 验证精度

**下一步**：打开 `02.03_chapter_test.ipynb` 完成课后测试！